In [1]:
# hide warnings
import warnings
warnings.filterwarnings('ignore')

# https://groups.csail.mit.edu/sls/downloads/restaurant/


In [21]:
import pandas as pd
import json
import requests


train = pd.read_csv('train.bio', sep='\t', header=None)
train.columns = ['tokens', 'ner_tags_str']

train.head(20)

data = open('train.bio', 'r').readlines()
train_tokens = []
train_tags = []

temp_tokens = []
temp_tags = []
for line in data:
    if line != '\n':
        tag, token = line.strip().split('\t')
        temp_tokens.append(token)
        temp_tags.append(tag)
    else:
        train_tokens.append(temp_tokens)
        train_tags.append(temp_tags)
        temp_tokens = []
        temp_tags = []


In [24]:
len(train_tokens), len(train_tags)


(7659, 7659)

In [25]:
from datasets import load_dataset, Dataset, DatasetDict

df = pd.DataFrame({'tokens': train_tokens, 'ner_tags_str': train_tags})
dataset = Dataset.from_pandas(df)
dataset = DatasetDict({'train': dataset, 'validation': dataset, 'test': dataset})
# dataset['train'][0]


In [26]:
dataset['train'][0]


unique_tags = set()
for tags in dataset['train']['ner_tags_str']:
    unique_tags.update(tags)


unique_tags = [x[2:] for x in list(unique_tags) if x != 'O']
unique_tags = list(set(unique_tags))

tag2index = {"O": 0}
for i, tag in enumerate(unique_tags):
    tag2index[f'B-{tag}'] = len(tag2index)
    tag2index[f'I-{tag}'] = len(tag2index)

index2tag = {v: k for k, v in tag2index.items()}


In [27]:
dataset = dataset.map(lambda example: {'ner_tags': [tag2index[tag] for tag in example['ner_tags_str']]})


Map:   0%|          | 0/7659 [00:00<?, ? examples/s]

Map:   0%|          | 0/7659 [00:00<?, ? examples/s]

Map:   0%|          | 0/7659 [00:00<?, ? examples/s]

In [28]:
dataset


DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags_str', 'ner_tags'],
        num_rows: 7659
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags_str', 'ner_tags'],
        num_rows: 7659
    })
    test: Dataset({
        features: ['tokens', 'ner_tags_str', 'ner_tags'],
        num_rows: 7659
    })
})

## Model Building

In [46]:
from transformers import AutoTokenizer


model_ckpt = 'huawei-noah/TinyBERT_General_4L_312D'
model_ckpt = 'distilbert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)


In [47]:
input = dataset['train'][0]['tokens']
input = tokenizer(input, is_split_into_words=True)
len(input.tokens()), len(input.word_ids())


(8, 8)

In [48]:
# input.tokens()


In [49]:
def align_labels_with_tokens(labels, word_ids):
  new_labels = []
  current_word=None
  for word_id in word_ids:
    if word_id != current_word:
      current_word = word_id
      label = -100 if word_id is None else labels[word_id]
      new_labels.append(label)

    elif word_id is None:
      new_labels.append(-100)

    else:
      label = labels[word_id]

      if label%2==1:
        label = label + 1
      new_labels.append(label)

  return new_labels


In [50]:
labels = dataset['train'][0]['ner_tags']
word_ids = input.word_ids()
# print(labels, word_ids)

# align_labels_with_tokens(labels, word_ids)




In [51]:
def tokenize_and_align_labels(examples):
  tokenized_inputs = tokenizer(examples['tokens'], truncation=True, is_split_into_words=True)

  all_labels = examples['ner_tags']

  new_labels = []
  for i, labels in enumerate(all_labels):
    word_ids = tokenized_inputs.word_ids(i)
    new_labels.append(align_labels_with_tokens(labels, word_ids))

  tokenized_inputs['labels'] = new_labels

  return tokenized_inputs


In [52]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)


Map:   0%|          | 0/7659 [00:00<?, ? examples/s]

Map:   0%|          | 0/7659 [00:00<?, ? examples/s]

Map:   0%|          | 0/7659 [00:00<?, ? examples/s]

## Data Collation and Metrics

In [53]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)


In [54]:
# !pip install seqeval
# !pip install evaluate

import evaluate
metric = evaluate.load('seqeval')
label_names = list(tag2index)

import numpy as np

def compute_metrics(eval_preds):
  logits, labels = eval_preds

  predictions = np.argmax(logits, axis=-1)

  true_labels = [[label_names[l] for l in label if l!=-100] for label in labels]

  true_predictions = [[label_names[p] for p,l in zip(prediction, label) if l!=-100]
                      for prediction, label in zip(predictions, labels)]

  all_metrics = metric.compute(predictions=true_predictions, references=true_labels)

  return {"precision": all_metrics['overall_precision'],
          "recall": all_metrics['overall_recall'],
          "f1": all_metrics['overall_f1'],
          "accuracy": all_metrics['overall_accuracy']}


In [55]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
                                                    model_ckpt,
                                                    id2label=index2tag,
                                                    label2id=tag2index)


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [56]:
model.config.num_labels


17

In [57]:
from transformers import TrainingArguments

args = TrainingArguments("finetuned-ner",
                         evaluation_strategy = "epoch",
                         save_strategy="epoch",
                         learning_rate = 2e-5,
                         num_train_epochs=3,
                         weight_decay=0.01)


In [58]:
from transformers import Trainer
trainer = Trainer(model=model,
                  args=args,
                  train_dataset = tokenized_datasets['train'],
                  eval_dataset = tokenized_datasets['validation'],
                  data_collator=data_collator,
                  compute_metrics=compute_metrics,
                  tokenizer=tokenizer)

trainer.train()


  0%|          | 0/2874 [00:00<?, ?it/s]

{'loss': 0.6611, 'grad_norm': 4.541174411773682, 'learning_rate': 1.65205288796103e-05, 'epoch': 0.52}


  0%|          | 0/958 [00:00<?, ?it/s]

{'eval_loss': 0.251808226108551, 'eval_precision': 0.771898918721905, 'eval_recall': 0.8271170995248324, 'eval_f1': 0.7985545954438336, 'eval_accuracy': 0.9255571298049549, 'eval_runtime': 5.5736, 'eval_samples_per_second': 1374.152, 'eval_steps_per_second': 171.881, 'epoch': 1.0}
{'loss': 0.339, 'grad_norm': 1.892743706703186, 'learning_rate': 1.30410577592206e-05, 'epoch': 1.04}
{'loss': 0.2568, 'grad_norm': 6.558841705322266, 'learning_rate': 9.561586638830899e-06, 'epoch': 1.57}


  0%|          | 0/958 [00:00<?, ?it/s]

{'eval_loss': 0.18774273991584778, 'eval_precision': 0.8326217980914113, 'eval_recall': 0.8632428562129792, 'eval_f1': 0.8476558754913555, 'eval_accuracy': 0.9439626064273513, 'eval_runtime': 5.7913, 'eval_samples_per_second': 1322.504, 'eval_steps_per_second': 165.421, 'epoch': 2.0}
{'loss': 0.2541, 'grad_norm': 1.7925233840942383, 'learning_rate': 6.082115518441197e-06, 'epoch': 2.09}
{'loss': 0.209, 'grad_norm': 2.6127052307128906, 'learning_rate': 2.6026443980514964e-06, 'epoch': 2.61}


  0%|          | 0/958 [00:00<?, ?it/s]

{'eval_loss': 0.16864076256752014, 'eval_precision': 0.8531877063754127, 'eval_recall': 0.8745687691206144, 'eval_f1': 0.8637459419497926, 'eval_accuracy': 0.9499609380172402, 'eval_runtime': 5.577, 'eval_samples_per_second': 1373.309, 'eval_steps_per_second': 171.776, 'epoch': 3.0}
{'train_runtime': 129.1617, 'train_samples_per_second': 177.893, 'train_steps_per_second': 22.251, 'train_loss': 0.32636656260108815, 'epoch': 3.0}


TrainOutput(global_step=2874, training_loss=0.32636656260108815, metrics={'train_runtime': 129.1617, 'train_samples_per_second': 177.893, 'train_steps_per_second': 22.251, 'total_flos': 105239751014754.0, 'train_loss': 0.32636656260108815, 'epoch': 3.0})

In [59]:
from transformers import pipeline

checkpoint = "finetuned-ner/checkpoint-2874"
token_classifier = pipeline(
    "token-classification", model=checkpoint, aggregation_strategy="simple"
)

token_classifier("which restaurant serves the best sushi in san francisco?")


[{'entity_group': 'Rating',
  'score': 0.9755293,
  'word': 'best',
  'start': 28,
  'end': 32},
 {'entity_group': 'Dish',
  'score': 0.87113434,
  'word': 'sushi',
  'start': 33,
  'end': 38},
 {'entity_group': 'Location',
  'score': 0.93644017,
  'word': 'san francisco',
  'start': 42,
  'end': 55}]